# Connections: standalone GINE baseline
Author: Will Blanton

Durable training workflow for the basic GINE tuning experiment. Frozen cached Qwen
embeddings → **trainable** LayerNorm and 1024→256 Linear/ReLU projection → candidate-local
GINE → mean/max pooling → learned readout. All active non-Qwen layers train together.
There is no global attention or frozen learned projection in the scoring path. The shared
checkpoint class retains unused inherited attention modules for initialization compatibility.

The original 15 pairwise edge channels are reused across subgraphs. Training uses the
probability mass of **any correct group**, uniformly mixing oracle s2/s3/s4 states.
Optuna selects by full-board validation **Hit@1**; MRR only breaks checkpoint ties.
See [the experiment](experiments/gine-hyperparameter-tuning.md) for the baseline evidence.
Run top to bottom to tune and retrain; weight saving remains the final cell.


## Imports and configuration


In [1]:
import ast
from copy import deepcopy
from datetime import datetime, timezone
import gc
import itertools
import json
from pathlib import Path
import random
import sys
from time import perf_counter

import numpy as np
import optuna
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from torch.utils.data import DataLoader

ROOT = Path.cwd()
if not (ROOT / "src/nyt_connections_solver").exists():
    raise RuntimeError("Open this notebook from the repository root")
sys.path.insert(0, str(ROOT / "src"))
from nyt_connections_solver.data_loader import collate_keep_wordlists, make_state_loaders
from nyt_connections_solver.graph_features import build_graph_features
from nyt_connections_solver.graph_models import StandaloneCandidateGINE

# Same train-only text initialization as the experiment; these weights are NOT frozen.
REFERENCE_PATH = Path("weights/experiments/20261023T150230Z-text-only.pt")
FEATURE_CACHE_DIR = Path("data/feature_cache/graph")
RUN_DIR = Path("experiments/notebook-gine-baseline")
RUN_DIR.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 16
OPTUNA_TRIALS = 16
TUNING_SEED = 0
SAMPLER_SEED = 20261025
TOTAL_UPDATES = 15_000
EVAL_EVERY = 250
PRUNE_AFTER = 8_000
FINAL_SEEDS = [10, 11, 12, 13]
RUN_VALIDATION_HARNESS = False
RUN_TEST_EVALUATION = False  # Explicit opt-in only; never used for model selection.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(TUNING_SEED)
print("Training device:", device)


Training device: cuda


## Fixed initialization, split, and cached features
The experiment started from a train-only text checkpoint. Reusing its initialization
and dates reproduces that training setup; it does not load tuned GINE weights.
Its LayerNorm/projection are updated throughout every trial and final retrain.
No test tensors are prepared unless explicitly requested after training.


In [2]:
reference = torch.load(REFERENCE_PATH, map_location="cpu", weights_only=True)
if reference["training_config"]["variant"] != "text-only":
    raise ValueError("Expected the shared train-only text initialization")
DATASET_PATH = Path(reference["dataset_path"])
split_dates = deepcopy(reference["split_dates"])
assert all(not (set(split_dates[a]) & set(split_dates[b]))
           for a, b in (("train", "val"), ("train", "test"), ("val", "test")))
connections_df = pd.read_csv(DATASET_PATH, index_col=0)
for row, value in {1298: "['line', 'plane', 'point', 'solid']",
                   1892: "['abyss', 'fly', 'matrix', 'thing']"}.items():
    if row in connections_df.index:
        connections_df.loc[row, "connections"] = value
connections_df = connections_df[~connections_df["date"].astype(str).str.contains("04-01")].copy()
connections_df["date"] = connections_df["date"].astype(str)
connections_df["connections"] = connections_df["connections"].apply(
    lambda value: ast.literal_eval(value) if isinstance(value, str) else value)
connections_df.reset_index(drop=True, inplace=True)

EMBEDDING_MODEL = reference["embedding_model_name"]
feature_options = {**reference["feature_metadata"]["options"], "use_spelling": False}
word_embedder = SentenceTransformer(EMBEDDING_MODEL, revision=feature_options["revision"], device=str(device))
word_embedder.requires_grad_(False)
word_embedder.eval()
base_feature_builder, graph_edge_builder, feature_metadata = build_graph_features(
    word_embedder, EMBEDDING_MODEL, cache_dir=FEATURE_CACHE_DIR, **feature_options)
assert feature_metadata == reference["feature_metadata"]
assert list(graph_edge_builder.feature_names) == reference["edge_feature_names"]
assert graph_edge_builder.dim == 15
print("Node width:", base_feature_builder.dim)
print("Edge channels:", graph_edge_builder.feature_names)

state_datasets = {}
for split in ("train", "val"):
    frame = connections_df[connections_df["date"].isin(split_dates[split])]
    if frame["date"].nunique() != len(split_dates[split]):
        raise ValueError(f"Dataset no longer reproduces {split} dates")
    state_datasets[split], _ = make_state_loaders(
        frame, word_embedder, None, BATCH_SIZE, shuffle=False,
        feature_builder=base_feature_builder, edge_providers=graph_edge_builder,
        cache=graph_edge_builder.cache)
# All training embeddings are now cached; free GPU memory for sequential trials.
word_embedder.to("cpu")
if device.type == "cuda":
    torch.cuda.empty_cache()

def training_loaders(seed):
    return {size: DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                            collate_fn=collate_keep_wordlists,
                            generator=torch.Generator().manual_seed(seed + size))
            for size, dataset in state_datasets["train"].items()}

val_state_loaders = {size: DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                                    collate_fn=collate_keep_wordlists)
                     for size, dataset in state_datasets["val"].items()}
print({split: {size: len(dataset) for size, dataset in datasets.items()}
       for split, datasets in state_datasets.items()})


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Node width: 1024
Edge channels: ('semantic_cosine', 'sense_jaccard', 'rare_concept_jaccard', 'longest_common_substring', 'common_prefix', 'common_suffix', 'prefix_semantic_similarity', 'suffix_semantic_similarity', 'shared_preceding_word', 'shared_following_word', 'exact_anagram', 'character_multiset_overlap', 'both_palindromes', 'both_initial_double', 'both_consecutive_double_pairs')
{'train': {2: 5808, 3: 3872, 4: 968}, 'val': {2: 726, 3: 484, 4: 121}}


## Probability-mass loss and ranking metrics
Hit@1 asks whether the highest ranked candidate is any true group. s4 is the primary
selection metric; s3/s2 report oracle reduced-state performance. Color diagnostics follow
the CSV group order Purple, Blue, Green, Yellow. Evaluation uses every validation state.


In [3]:
def make_artifacts() -> dict[int, dict[str, torch.Tensor]]:
    artifacts = {}
    for size in (2, 3, 4):
        candidates = list(itertools.combinations(range(size * 4), 4))
        true_groups = {tuple(range(group * 4, group * 4 + 4)) for group in range(size)}
        relevance = [int(candidate in true_groups) for candidate in candidates]
        artifacts[size] = {
            "idx": torch.tensor(candidates, dtype=torch.long),
            "y": torch.tensor(relevance, dtype=torch.bool),
        }
        true_groups_tensor = artifacts[size]["idx"][artifacts[size]["y"]]
        overlaps = (artifacts[size]["idx"][:, None, :, None] ==
                    true_groups_tensor[None, :, None, :]).any(dim=-1).sum(dim=-1)
        artifacts[size]["near_miss"] = (overlaps == 3).any(dim=1) & ~artifacts[size]["y"]
    return artifacts

def state_loss(scores: torch.Tensor, relevance: torch.Tensor) -> torch.Tensor:
    logits = scores.squeeze(-1).float()
    positive = relevance.unsqueeze(0).expand(len(logits), -1)
    log_probs = torch.log_softmax(logits, dim=1)
    masked = torch.where(positive, log_probs, torch.full_like(log_probs, float("-inf")))
    return -torch.logsumexp(masked, dim=1).mean()

def ranking_metrics(scores: torch.Tensor, relevance: torch.Tensor) -> dict[str, torch.Tensor]:
    logits = scores.squeeze(-1).float()
    positive = relevance.unsqueeze(0).expand(len(logits), -1)
    order = logits.argsort(dim=1, descending=True)
    ranked = positive.gather(1, order)
    first_rank = ranked.float().argmax(dim=1) + 1
    ranks = torch.arange(1, logits.shape[1] + 1, device=logits.device, dtype=torch.float)
    precision = ranked.float().cumsum(dim=1) / ranks
    average_precision = (precision * ranked).sum(dim=1) / positive.sum(dim=1)
    cutoff = min(4, logits.shape[1])
    discounts = torch.log2(torch.arange(2, cutoff + 2, device=logits.device, dtype=torch.float))
    dcg = (ranked[:, :cutoff].float() / discounts).sum(dim=1)
    ideal_count = torch.minimum(positive.sum(dim=1), torch.tensor(cutoff, device=logits.device))
    ideal = torch.stack([
        (torch.ones(int(count.item()), device=logits.device) / discounts[: int(count.item())]).sum()
        for count in ideal_count
    ])
    return {
        "hit1": ranked[:, 0].float(),
        "mrr": first_rank.float().reciprocal(),
        "hit4": ranked[:, :cutoff].any(dim=1).float(),
        "map": average_precision,
        "ndcg4": dcg / ideal,
    }


In [4]:
def score_batch(
    model: StandaloneCandidateGINE,
    batch: dict,
    group_idx: torch.Tensor,
    device: torch.device,
) -> torch.Tensor:
    return model(
        batch["x"].to(device),
        group_idx=group_idx,
        edge_features=batch["edge_features"].to(device),
    )

def evaluate(
    model: StandaloneCandidateGINE,
    loaders: dict[int, DataLoader],
    artifacts: dict[int, dict[str, torch.Tensor]],
    device: torch.device,
    *, by_color: bool = False,
) -> dict[int, dict[str, float]]:
    model.eval()
    results = {}
    with torch.inference_mode():
        for size, loader in loaders.items():
            group_idx = artifacts[size]["idx"].to(device)
            relevance = artifacts[size]["y"].to(device)
            totals = {key: 0.0 for key in ("hit1", "mrr", "hit4", "map", "ndcg4")}
            count = 0
            color_rows = {color: [] for color in ("purple", "blue", "green", "yellow")}
            for batch in loader:
                scores = score_batch(model, batch, group_idx, device)
                metrics = ranking_metrics(scores, relevance)
                for key, values in metrics.items():
                    totals[key] += values.sum().item()
                if by_color:
                    # True groups occupy consecutive blocks; source_group_idx
                    # identifies their original CSV rows, including partial states.
                    logits = scores.squeeze(-1)
                    order = logits.argsort(dim=1, descending=True)
                    ranks = order.argsort(dim=1) + 1
                    targets = relevance.nonzero().flatten()
                    target_ranks = ranks[:, targets].cpu()
                    for source_groups, row in zip(batch["source_group_idx"], target_ranks):
                        for source, rank in zip(source_groups.tolist(), row.tolist()):
                            color_rows[("purple", "blue", "green", "yellow")[source]].append(rank)
                count += len(scores)
            if count == 0:
                raise ValueError(f"Validation loader s{size} is empty")
            results[size] = {key: value / count for key, value in totals.items()}
            if by_color:
                results[size]["by_color"] = {
                    color: {
                        "groups": len(rows),
                        "mean_rank": float(np.mean(rows)),
                        "mrr": float(np.mean([1 / rank for rank in rows])),
                        "top1": float(np.mean([rank == 1 for rank in rows])),
                        "top4": float(np.mean([rank <= 4 for rank in rows])),
                    }
                    for color, rows in color_rows.items() if rows
                }
    return results

def next_batch(iterators: dict, loaders: dict, size: int) -> dict:
    try:
        return next(iterators[size])
    except StopIteration:
        iterators[size] = iter(loaders[size])
        return next(iterators[size])


In [5]:
state_artifacts = make_artifacts()


## Model construction and training loop
Constant LR AdamW, no scheduler or auxiliary loss. Each trial gets up to 15,000 updates
(about 82 s4, 20.7 s3 and 13.8 s2 passes for the current data). No ordinary early stopping.
Restore the highest validation s4 Hit@1 checkpoint, using MRR only for ties.


In [6]:
def suggest(trial):
    return dict(hidden_width=trial.suggest_categorical('hidden_width', [32, 64, 128]),
                gine_layers=trial.suggest_int('gine_layers', 1, 3),
                readout_width=trial.suggest_categorical('readout_width', [64, 128, 256]),
                dropout=trial.suggest_categorical('dropout', [0., .1, .2]),
                lr=trial.suggest_float('lr', 3e-5, 5e-4, log=True),
                weight_decay=trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True))

def build_model(reference, params, idx, seed):
    seed_everything(seed)
    config = deepcopy(reference['model_config'])
    config.update(edge_hidden=params['hidden_width'], gine_layers=params['gine_layers'],
                  readout_width=params['readout_width'], gine_dropout=params['dropout'],
                  edge_bias_scale=0., train_text_projection=True, train_input_norm=True)
    model = StandaloneCandidateGINE(group_idx=idx, **config)
    missing, unexpected = model.load_state_dict(reference['model_state_dict'], strict=False)
    if unexpected or any(not n.startswith(('edge_encoder.', 'edge_gate.', 'edge_correction.', 'group_readout.'))
                         for n in missing):
        raise ValueError('Unexpected mismatch loading the shared text-only initialization')
    return model, config

BASELINE_PARAMS = dict(hidden_width=32, gine_layers=2, readout_width=128,
                       dropout=0., lr=1e-4, weight_decay=1e-6)
PREVIOUS_BEST_PARAMS = dict(hidden_width=128, gine_layers=2, readout_width=128,
                            dropout=0., lr=0.0003016684667937828,
                            weight_decay=9.500890359387789e-05)
# Previous winner is recorded for comparison, not enqueued into the original search.


In [7]:
def train_gine(model, params, seed, trial=None):
    loaders = training_loaders(seed)
    model = model.to(device)
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),
                                  lr=params["lr"], weight_decay=params["weight_decay"])
    scaler = torch.amp.GradScaler(device.type, enabled=device.type == "cuda")
    sizes = tuple(sorted(loaders))
    iterators = {size: iter(loader) for size, loader in loaders.items()}
    artifacts = {size: {key: value.to(device) for key, value in values.items()}
                 for size, values in state_artifacts.items()}
    pending_sizes = []
    best_key = (-float("inf"), -float("inf"))
    best_state = None
    best_update = 0
    # Match the experiment's update-0 validation checkpoint as well.
    baseline_metrics = evaluate(model, val_state_loaders, state_artifacts, device, by_color=True)
    best_key = (baseline_metrics[4]["hit1"], baseline_metrics[4]["mrr"])
    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    history = []
    running_loss = 0.
    interval_updates = 0
    started = perf_counter()
    for step in range(1, TOTAL_UPDATES + 1):
        if not pending_sizes:
            pending_sizes = list(sizes)
            random.shuffle(pending_sizes)
        size = pending_sizes.pop()
        batch = next_batch(iterators, loaders, size)
        artifact = artifacts[size]
        model.train()
        optimizer.zero_grad(set_to_none=True)
        scores = score_batch(model, batch, artifact["idx"], device)
        loss = state_loss(scores, artifact["y"])
        if not torch.isfinite(loss):
            raise ValueError(f"Nonfinite loss at update {step}")
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
        interval_updates += 1
        if step % EVAL_EVERY and step != TOTAL_UPDATES:
            continue
        val_started = perf_counter()
        metrics = evaluate(model, val_state_loaders, state_artifacts, device)
        val_seconds = perf_counter() - val_started
        key = (metrics[4]["hit1"], metrics[4]["mrr"])
        if key > best_key:
            best_key, best_update = key, step
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        row = dict(update=step, train_loss=running_loss / interval_updates,
                   best_update=best_update, learning_rate=optimizer.param_groups[0]["lr"],
                   train_seconds=perf_counter() - started - val_seconds, validation_seconds=val_seconds,
                   **{f"val_s{s}_{m}": v for s, values in metrics.items() for m, v in values.items()})
        history.append(row)
        print(f"Update {step}/{TOTAL_UPDATES} | loss {row['train_loss']:.4f} | "
              f"s4 Hit@1 {metrics[4]['hit1']:.4f}, MRR {metrics[4]['mrr']:.4f} | "
              f"s3 {metrics[3]['hit1']:.4f} | s2 {metrics[2]['hit1']:.4f} | "
              f"best {best_key[0]:.4f} @ {best_update}", flush=True)
        if trial is not None:
            trial.report(best_key[0], step)
            trial.set_user_attr("updates_ran", step)
            trial.set_user_attr("best_hit1", best_key[0])
            if step >= PRUNE_AFTER and trial.should_prune():
                raise optuna.TrialPruned(f"Pruned at {step}: best Hit@1 {best_key[0]:.4f}")
        running_loss, interval_updates = 0., 0
        started = perf_counter()
    if best_state is None:
        raise ValueError("No validation checkpoint recorded")
    model.load_state_dict(best_state)
    metrics = evaluate(model, val_state_loaders, state_artifacts, device, by_color=True)
    summary = dict(selection_metric="validation_s4_hit1", tie_break_metric="validation_s4_mrr",
                   best_update=best_update, updates_ran=step,
                   validation={f"s{s}": values for s, values in metrics.items()}, history=history,
                   baseline_validation=baseline_metrics,
                   validation_macro={m: float(np.mean([metrics[s][m] for s in sizes]))
                                     for m in ("hit1", "mrr", "hit4", "map", "ndcg4")})
    return model.to("cpu").eval(), summary


## Optuna: same search space and conservative pruning
16 sequential trials, training seed 0, TPE seed 20261025 with six startup trials.
Median pruning starts at update 8,000, checks every 1,000 updates, and requires four
comparable trials. It receives best-so-far s4 Hit@1. No harness runs inside tuning.
The SQLite study lives separately from the completed command-line experiment and resumes
finished trials. Change `RUN_DIR` to start a new study with a different setup.


In [8]:
if OPTUNA_TRIALS < 6 or TOTAL_UPDATES < 10_000 or not (8_000 <= PRUNE_AFTER < TOTAL_UPDATES):
    raise ValueError("Require >=6 trials, >=10000 updates, and pruning no earlier than 8000")
study_manifest = dict(reference=str(REFERENCE_PATH), model_class="StandaloneCandidateGINE",
                      model_config=reference["model_config"], feature_metadata=feature_metadata,
                      split_dates=split_dates, batch_size=BATCH_SIZE,
                      updates=TOTAL_UPDATES, eval_every=EVAL_EVERY, prune_after=PRUNE_AFTER,
                      tuning_seed=TUNING_SEED, sampler_seed=SAMPLER_SEED,
                      metric="validation_s4_hit1", search_space_version=1)
manifest_path = RUN_DIR / "manifest.json"
if manifest_path.exists() and json.loads(manifest_path.read_text()) != study_manifest:
    raise ValueError("RUN_DIR contains a different setup; choose a new directory")
manifest_path.write_text(json.dumps(study_manifest, indent=2) + "\n")
study = optuna.create_study(
    study_name="notebook_gine_baseline", storage=f"sqlite:///{(RUN_DIR / 'study.sqlite3').resolve()}",
    direction="maximize", load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SAMPLER_SEED, n_startup_trials=6),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=6, n_warmup_steps=PRUNE_AFTER,
                                      interval_steps=1000, n_min_trials=4))
if not study.trials:
    study.enqueue_trial(BASELINE_PARAMS)
    study.enqueue_trial({**BASELINE_PARAMS, "hidden_width": 64})
    study.enqueue_trial({**BASELINE_PARAMS, "hidden_width": 64, "dropout": .1, "weight_decay": 1e-4})

def objective(trial):
    params = suggest(trial)
    trial_model, _ = build_model(reference, params, state_artifacts[4]["idx"], TUNING_SEED)
    try:
        trained, summary = train_gine(trial_model, params, TUNING_SEED, trial=trial)
        trial.set_user_attr("best_update", summary["best_update"])
        (RUN_DIR / f"trial-{trial.number:02d}.json").write_text(json.dumps(
            dict(params=params, training=summary), indent=2) + "\n")
        return summary["validation"]["s4"]["hit1"]
    finally:
        trial_model.to("cpu")
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

finished = sum(trial.state.is_finished() for trial in study.trials)
study.optimize(objective, n_trials=max(0, OPTUNA_TRIALS - finished), gc_after_trial=True)
best_params = dict(study.best_params)
study.trials_dataframe().to_csv(RUN_DIR / "trials.csv", index=False)
(RUN_DIR / "best.json").write_text(json.dumps(
    dict(trial=study.best_trial.number, value=study.best_value, params=best_params), indent=2) + "\n")
print("Selected validation Hit@1:", study.best_value)
print("Selected parameters:", best_params)
display(study.trials_dataframe())


[I 2026-10-25 21:08:34,696] A new study created in RDB with name: notebook_gine_baseline


Update 250/15000 | loss 4.6062 | s4 Hit@1 0.0826, MRR 0.1374 | s3 0.1157 | s2 0.2080 | best 0.0826 @ 250
Update 500/15000 | loss 3.8445 | s4 Hit@1 0.0992, MRR 0.1709 | s3 0.1364 | s2 0.2052 | best 0.0992 @ 500
Update 750/15000 | loss 3.5823 | s4 Hit@1 0.1322, MRR 0.1937 | s3 0.1570 | s2 0.2245 | best 0.1322 @ 750
Update 1000/15000 | loss 3.1302 | s4 Hit@1 0.0909, MRR 0.1676 | s3 0.1322 | s2 0.2342 | best 0.1322 @ 750
Update 1250/15000 | loss 2.7835 | s4 Hit@1 0.1405, MRR 0.2156 | s3 0.1839 | s2 0.2617 | best 0.1405 @ 1250
Update 1500/15000 | loss 2.4033 | s4 Hit@1 0.1488, MRR 0.2345 | s3 0.1963 | s2 0.2782 | best 0.1488 @ 1500
Update 1750/15000 | loss 2.0834 | s4 Hit@1 0.1736, MRR 0.2495 | s3 0.2087 | s2 0.2851 | best 0.1736 @ 1750
Update 2000/15000 | loss 1.8597 | s4 Hit@1 0.1405, MRR 0.2335 | s3 0.1839 | s2 0.2782 | best 0.1736 @ 1750
Update 2250/15000 | loss 1.6967 | s4 Hit@1 0.1405, MRR 0.2450 | s3 0.1983 | s2 0.2893 | best 0.1736 @ 1750
Update 2500/15000 | loss 1.5121 | s4 Hit@1 0

[I 2026-10-25 21:09:32,796] Trial 0 finished with value: 0.23140495867768596 and parameters: {'hidden_width': 32, 'gine_layers': 2, 'readout_width': 128, 'dropout': 0.0, 'lr': 0.0001, 'weight_decay': 1e-06}. Best is trial 0 with value: 0.23140495867768596.


Update 250/15000 | loss 4.4168 | s4 Hit@1 0.1240, MRR 0.1755 | s3 0.1467 | s2 0.2287 | best 0.1240 @ 250
Update 500/15000 | loss 3.5346 | s4 Hit@1 0.1157, MRR 0.1917 | s3 0.1529 | s2 0.2438 | best 0.1240 @ 250
Update 750/15000 | loss 3.1676 | s4 Hit@1 0.1405, MRR 0.2270 | s3 0.1756 | s2 0.2658 | best 0.1405 @ 750
Update 1000/15000 | loss 2.5425 | s4 Hit@1 0.1736, MRR 0.2614 | s3 0.2149 | s2 0.3003 | best 0.1736 @ 1000
Update 1250/15000 | loss 2.2058 | s4 Hit@1 0.1818, MRR 0.2870 | s3 0.2355 | s2 0.3388 | best 0.1818 @ 1250
Update 1500/15000 | loss 1.8442 | s4 Hit@1 0.2066, MRR 0.3071 | s3 0.2562 | s2 0.3609 | best 0.2066 @ 1500
Update 1750/15000 | loss 1.5551 | s4 Hit@1 0.1901, MRR 0.2956 | s3 0.2355 | s2 0.3168 | best 0.2066 @ 1500
Update 2000/15000 | loss 1.3986 | s4 Hit@1 0.2066, MRR 0.3247 | s3 0.2541 | s2 0.3347 | best 0.2066 @ 2000
Update 2250/15000 | loss 1.2216 | s4 Hit@1 0.2562, MRR 0.3501 | s3 0.2789 | s2 0.3430 | best 0.2562 @ 2250
Update 2500/15000 | loss 1.0593 | s4 Hit@1 

[I 2026-10-25 21:10:51,715] Trial 1 finished with value: 0.2727272727272727 and parameters: {'hidden_width': 64, 'gine_layers': 2, 'readout_width': 128, 'dropout': 0.0, 'lr': 0.0001, 'weight_decay': 1e-06}. Best is trial 1 with value: 0.2727272727272727.


Update 250/15000 | loss 4.7230 | s4 Hit@1 0.0496, MRR 0.1085 | s3 0.0992 | s2 0.1983 | best 0.0496 @ 250
Update 500/15000 | loss 3.9814 | s4 Hit@1 0.0992, MRR 0.1774 | s3 0.1570 | s2 0.2355 | best 0.0992 @ 500
Update 750/15000 | loss 3.4990 | s4 Hit@1 0.1736, MRR 0.2414 | s3 0.2025 | s2 0.2782 | best 0.1736 @ 750
Update 1000/15000 | loss 3.0177 | s4 Hit@1 0.1983, MRR 0.2755 | s3 0.2149 | s2 0.2920 | best 0.1983 @ 1000
Update 1250/15000 | loss 2.5542 | s4 Hit@1 0.2231, MRR 0.2986 | s3 0.2190 | s2 0.2796 | best 0.2231 @ 1250
Update 1500/15000 | loss 2.1848 | s4 Hit@1 0.1983, MRR 0.3009 | s3 0.2335 | s2 0.3085 | best 0.2231 @ 1250
Update 1750/15000 | loss 1.8521 | s4 Hit@1 0.2314, MRR 0.3094 | s3 0.2417 | s2 0.3278 | best 0.2314 @ 1750
Update 2000/15000 | loss 1.6161 | s4 Hit@1 0.2314, MRR 0.3048 | s3 0.2376 | s2 0.2920 | best 0.2314 @ 1750
Update 2250/15000 | loss 1.4743 | s4 Hit@1 0.1653, MRR 0.2708 | s3 0.2087 | s2 0.3154 | best 0.2314 @ 1750
Update 2500/15000 | loss 1.2922 | s4 Hit@1 

... (25 outputs left out)

Update 250/15000 | loss 4.4998 | s4 Hit@1 0.1405, MRR 0.2072 | s3 0.1612 | s2 0.2452 | best 0.1405 @ 250
Update 500/15000 | loss 3.4492 | s4 Hit@1 0.2727, MRR 0.3656 | s3 0.2872 | s2 0.3595 | best 0.2727 @ 500
Update 750/15000 | loss 3.0147 | s4 Hit@1 0.2397, MRR 0.3501 | s3 0.2748 | s2 0.3650 | best 0.2727 @ 500
Update 1000/15000 | loss 2.7061 | s4 Hit@1 0.2893, MRR 0.3796 | s3 0.3120 | s2 0.3747 | best 0.2893 @ 1000
Update 1250/15000 | loss 2.5121 | s4 Hit@1 0.2479, MRR 0.3592 | s3 0.2872 | s2 0.3636 | best 0.2893 @ 1000
Update 1500/15000 | loss 2.3407 | s4 Hit@1 0.3140, MRR 0.4117 | s3 0.3285 | s2 0.3871 | best 0.3140 @ 1500
Update 1750/15000 | loss 2.1626 | s4 Hit@1 0.2975, MRR 0.3930 | s3 0.3161 | s2 0.3747 | best 0.3140 @ 1500
Update 2000/15000 | loss 2.0225 | s4 Hit@1 0.3140, MRR 0.3931 | s3 0.3223 | s2 0.3815 | best 0.3140 @ 1500
Update 2250/15000 | loss 1.8529 | s4 Hit@1 0.2893, MRR 0.3825 | s3 0.3058 | s2 0.3760 | best 0.3140 @ 1500
Update 2500/15000 | loss 1.7433 | s4 Hit@1 

[I 2026-10-25 21:30:25,920] Trial 15 finished with value: 0.3140495867768595 and parameters: {'hidden_width': 32, 'gine_layers': 3, 'readout_width': 64, 'dropout': 0.1, 'lr': 0.00042823372828683414, 'weight_decay': 0.0014714978756625477}. Best is trial 4 with value: 0.35537190082644626.


Selected validation Hit@1: 0.35537190082644626
Selected parameters: {'hidden_width': 128, 'gine_layers': 2, 'readout_width': 128, 'dropout': 0.0, 'lr': 0.0003392786247637008, 'weight_decay': 0.0037488737958182063}


,number,value,datetime_start,datetime_complete,duration,params_dropout,params_gine_layers,params_hidden_width,params_lr,params_readout_width,params_weight_decay,user_attrs_best_hit1,user_attrs_best_update,user_attrs_updates_ran,system_attrs_fixed_params,system_attrs_tpe:relative_params:0,state
0,0,0.231405,2026-10-25 21:08:34.774387,2026-10-25 21:09:32.775109,0 days 00:00:58.000722,0.0,2,32,0.000100,128,0.000001,0.231405,13500.0,15000,"{'hidden_width': 32, 'gine_layers': 2, 'readou...",NaN,COMPLETE
1,1,0.272727,2026-10-25 21:09:32.988594,2026-10-25 21:10:51.694033,0 days 00:01:18.705439,0.0,2,64,0.000100,128,0.000001,0.272727,4250.0,15000,"{'hidden_width': 64, 'gine_layers': 2, 'readou...",NaN,COMPLETE
2,2,0.247934,2026-10-25 21:10:51.906401,2026-10-25 21:12:09.546850,0 days 00:01:17.640449,0.1,2,64,0.000100,128,0.000100,0.247934,14000.0,15000,"{'hidden_width': 64, 'gine_layers': 2, 'readou...",NaN,COMPLETE
3,3,0.297521,2026-10-25 21:12:09.747647,2026-10-25 21:13:19.053221,0 days 00:01:09.305574,0.1,3,32,0.000216,64,0.000004,0.297521,14250.0,15000,NaN,NaN,COMPLETE
4,4,0.355372,2026-10-25 21:13:19.260462,2026-10-25 21:15:21.462191,0 days 00:02:02.201729,0.0,2,128,0.000339,128,0.003749,0.355372,2750.0,15000,NaN,NaN,COMPLETE
5,5,0.247934,2026-10-25 21:15:21.704047,2026-10-25 21:17:30.256543,0 days 00:02:08.552496,0.2,2,128,0.000038,256,0.000005,0.247934,6000.0,15000,NaN,NaN,COMPLETE
6,6,0.289256,2026-10-25 21:17:30.463361,2026-10-25 21:18:53.342588,0 days 00:01:22.879227,0.0,1,128,0.000383,128,0.000168,0.289256,1000.0,15000,NaN,"{""dropout"": 0.0, ""gine_layers"": 1, ""hidden_wid...",COMPLETE
7,7,0.297521,2026-10-25 21:18:53.545750,2026-10-25 21:21:34.522356,0 days 00:02:40.976606,0.0,3,128,0.000139,128,0.004159,0.297521,5250.0,15000,NaN,"{""dropout"": 0.0, ""gine_layers"": 3, ""hidden_wid...",COMPLETE
8,8,0.264463,2026-10-25 21:21:34.732784,2026-10-25 21:22:59.561407,0 days 00:01:24.828623,0.0,3,128,0.000032,64,0.000682,0.264463,NaN,8000,NaN,"{""dropout"": 0.0, ""gine_layers"": 3, ""hidden_wid...",PRUNED
9,9,0.214876,2026-10-25 21:22:59.757027,2026-10-25 21:23:40.729629,0 days 00:00:40.972602,0.0,1,128,0.000046,128,0.000254,0.214876,NaN,8000,NaN,"{""dropout"": 0.0, ""gine_layers"": 1, ""hidden_wid...",PRUNED


## Full retraining with the selected parameters
Fresh seeds 10–13, each trained for all 15,000 updates, starting from the same train-only
text initialization. No pruning in final retrains. The input norm/projection and all active
GINE/readout parameters remain trainable. Report the mean across seeds; choose the primary
saved model by validation s4 Hit@1 (MRR breaks ties), never by test performance.


In [9]:
final_runs = []
for seed in FINAL_SEEDS:
    final_model, model_config = build_model(reference, best_params, state_artifacts[4]["idx"], seed)
    assert all(p.requires_grad for module in (final_model.input_norm, final_model.proj,
                                              final_model.edge_encoder, final_model.group_readout)
               for p in module.parameters())
    final_model, summary = train_gine(final_model, best_params, seed)
    final_runs.append(dict(seed=seed, model=final_model, model_config=model_config, training=summary))
    (RUN_DIR / f"final-seed-{seed}.json").write_text(json.dumps(
        dict(seed=seed, params=best_params, training=summary), indent=2) + "\n")
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

primary_run = max(final_runs, key=lambda run: (
    run["training"]["validation"]["s4"]["hit1"], run["training"]["validation"]["s4"]["mrr"]))
model = primary_run["model"]
best_model_config = primary_run["model_config"]
best_training_config = dict(
    variant="gine-basic-tuning", params=best_params, seed=primary_run["seed"],
    reference_checkpoint=str(REFERENCE_PATH), updates=TOTAL_UPDATES, batch_size=BATCH_SIZE,
    metric="s4 Hit@1", auxiliary_loss=False, lr_schedule=False,
    total_trials=OPTUNA_TRIALS, eval_every=EVAL_EVERY, prune_after=PRUNE_AFTER,
    sampler_seed=SAMPLER_SEED, tuning_seed=TUNING_SEED)
ranking_rows = [dict(seed=run["seed"], selected_update=run["training"]["best_update"],
                     **{f"{size}_{metric}": values[metric]
                        for size, values in run["training"]["validation"].items()
                        for metric in ("hit1", "mrr", "hit4")}) for run in final_runs]
ranking_results = pd.DataFrame(ranking_rows).set_index("seed")
display(ranking_results)
display(ranking_results.drop(columns="selected_update").mean().to_frame("seed_mean"))
print("Primary seed selected on validation:", primary_run["seed"])
display(pd.DataFrame(primary_run["training"]["validation"]["s4"]["by_color"]).T)


Update 250/15000 | loss 4.0613 | s4 Hit@1 0.1240, MRR 0.2051 | s3 0.1508 | s2 0.2590 | best 0.1240 @ 250
Update 500/15000 | loss 3.0519 | s4 Hit@1 0.2975, MRR 0.3882 | s3 0.3409 | s2 0.4050 | best 0.2975 @ 500
Update 750/15000 | loss 2.6922 | s4 Hit@1 0.2975, MRR 0.4162 | s3 0.3409 | s2 0.4132 | best 0.2975 @ 750
Update 1000/15000 | loss 2.4265 | s4 Hit@1 0.3471, MRR 0.4255 | s3 0.3574 | s2 0.4242 | best 0.3471 @ 1000
Update 1250/15000 | loss 2.3492 | s4 Hit@1 0.2975, MRR 0.4196 | s3 0.3326 | s2 0.4105 | best 0.3471 @ 1000
Update 1500/15000 | loss 2.2812 | s4 Hit@1 0.3140, MRR 0.4114 | s3 0.3450 | s2 0.4160 | best 0.3471 @ 1000
Update 1750/15000 | loss 2.1268 | s4 Hit@1 0.3471, MRR 0.4435 | s3 0.3657 | s2 0.4325 | best 0.3471 @ 1750
Update 2000/15000 | loss 2.0598 | s4 Hit@1 0.3140, MRR 0.4257 | s3 0.3719 | s2 0.4435 | best 0.3471 @ 1750
Update 2250/15000 | loss 1.9151 | s4 Hit@1 0.3223, MRR 0.4084 | s3 0.3533 | s2 0.4270 | best 0.3471 @ 1750
Update 2500/15000 | loss 1.7997 | s4 Hit@1 

,selected_update,s2_hit1,s2_mrr,s2_hit4,s3_hit1,s3_mrr,s3_hit4,s4_hit1,s4_mrr,s4_hit4
seed,,,,,,,,,,
10,2750,0.421488,0.550195,0.668044,0.359504,0.469423,0.537190,0.347107,0.452904,0.520661
11,1500,0.435262,0.557312,0.657025,0.363636,0.468409,0.549587,0.347107,0.440581,0.520661
12,1750,0.422865,0.544183,0.648760,0.376033,0.482468,0.564050,0.347107,0.458977,0.545455
13,6000,0.414601,0.539507,0.639118,0.359504,0.464981,0.549587,0.338843,0.442563,0.537190


,seed_mean
s2_hit1,0.423554
s2_mrr,0.547799
s2_hit4,0.653237
s3_hit1,0.364669
s3_mrr,0.471320
s3_hit4,0.550103
s4_hit1,0.345041
s4_mrr,0.448756
s4_hit4,0.530992


Primary seed selected on validation: 12


,groups,mean_rank,mrr,top1,top4
purple,121.0,925.776860,0.031229,0.024793,0.033058
blue,121.0,644.776860,0.067467,0.033058,0.090909
green,121.0,438.983471,0.147342,0.115702,0.165289
yellow,121.0,265.371901,0.248404,0.173554,0.297521


## Optional final evaluation
Validation feedback search uses beam width 20, full-board scoring and four mistakes.
It never feeds back into training or Optuna. Test ranking is explicitly opt-in and uses the
already selected primary model. Existing test puzzles have been inspected previously;
new puzzles should supply the eventual final holdout.


In [12]:
from nyt_connections_solver.harness import SearchHarness
from nyt_connections_solver.feedback_search import FeedbackSearchHarness
from nyt_connections_solver.evaluation.feedback import evaluate_feedback_games
from nyt_connections_solver.evaluation.search import Puzzle
puzzles = []
for date in split_dates["val"]:
    groups = tuple(tuple(group) for group in connections_df.loc[
        connections_df["date"] == date, "connections"])
    if len(groups) != 4:
        raise ValueError(f"Expected four groups for {date}")
    puzzles.append(Puzzle(date, tuple(word for group in groups for word in group),
                            frozenset(map(frozenset, groups)), groups))
validation_harness_results = []
for run in final_runs:
    harness = SearchHarness.from_model(
        run["model"], word_embedder, input_dim=base_feature_builder.dim, device=device,
        node_feature_builder=base_feature_builder, edge_feature_builder=graph_edge_builder.encode,
        beam_width=20, state_batch_size=4)
    policy = FeedbackSearchHarness.from_harness(harness, beam_width=20, score_mode="full_board")
    result = evaluate_feedback_games(policy, puzzles, max_mistakes=4, seed=0)
    result.pop("games")
    validation_harness_results.append(dict(seed=run["seed"], **result))
    run["model"].to("cpu")
    del policy, harness
display(pd.DataFrame(validation_harness_results).set_index("seed"))

,puzzles,solved,solve_rate,mean_correct_groups,first_guess_accuracy,mean_guesses,mean_mistakes,one_away_guesses,correct_groups_histogram,total_seconds,mean_seconds
seed,,,,,,,,,,,
10,121,29,0.239669,1.561983,0.404959,5.033058,3.471074,258,"{0: 39, 1: 33, 2: 20, 3: 0, 4: 29}",31.400990,0.259512
11,121,32,0.264463,1.595041,0.371901,4.983471,3.388430,249,"{0: 43, 1: 27, 2: 19, 3: 0, 4: 32}",31.171021,0.257612
12,121,30,0.247934,1.611570,0.396694,5.008264,3.396694,231,"{0: 38, 1: 31, 2: 22, 3: 0, 4: 30}",30.294381,0.250367
13,121,32,0.264463,1.586777,0.404959,5.033058,3.446281,238,"{0: 39, 1: 36, 2: 14, 3: 0, 4: 32}",30.737239,0.254027


In [13]:
test_datasets, _ = make_state_loaders(
    connections_df[connections_df["date"].isin(split_dates["test"])], word_embedder,
    None, BATCH_SIZE, shuffle=False, feature_builder=base_feature_builder,
    edge_providers=graph_edge_builder, cache=graph_edge_builder.cache)
test_loaders = {s: DataLoader(d, batch_size=BATCH_SIZE, shuffle=False,
                                collate_fn=collate_keep_wordlists) for s, d in test_datasets.items()}
test_metrics = evaluate(model.to(device), test_loaders, state_artifacts, device, by_color=True)
model.to("cpu")
print("Primary checkpoint test metrics:", test_metrics)

Primary checkpoint test metrics: {2: {'hit1': 0.40633608815426997, 'mrr': 0.5304674603065511, 'hit4': 0.6308539944903582, 'map': 0.3101730673437933, 'ndcg4': 0.33516042997686, 'by_color': {'purple': {'groups': 363, 'mean_rank': 39.955922865013775, 'mrr': 0.0920075580010562, 'top1': 0.0440771349862259, 'top4': 0.07988980716253444}, 'blue': {'groups': 363, 'mean_rank': 27.12947658402204, 'mrr': 0.21259285886140353, 'top1': 0.14325068870523416, 'top4': 0.22865013774104684}, 'green': {'groups': 363, 'mean_rank': 18.096418732782368, 'mrr': 0.35565177154846533, 'top1': 0.24517906336088155, 'top4': 0.44352617079889806}, 'yellow': {'groups': 363, 'mean_rank': 10.898071625344352, 'mrr': 0.490561398921286, 'top1': 0.38016528925619836, 'top4': 0.5757575757575758}}}, 3: {'hit1': 0.3285123966942149, 'mrr': 0.4495077490313979, 'hit4': 0.5599173553719008, 'map': 0.17721993002024564, 'ndcg4': 0.21992951994838794, 'by_color': {'purple': {'groups': 363, 'mean_rank': 277.08815426997245, 'mrr': 0.02570370

## Save GINE weights
Save every fresh-seed retrain, plus identify the primary checkpoint chosen on validation.
Each file includes the complete trainable model, exact configuration, features and split dates.
Reload with `nyt_connections_solver.checkpoints.load_graph_harness(weights_path)`;
the Streamlit diagnostic app uses the same loader.


In [11]:
weights_dir = Path("weights")
weights_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
saved_runs = []
for run in final_runs:
    run_weights_path = weights_dir / f"standalone_gine_optuna_text_seed_{run['seed']}_{timestamp}.pt"
    training_config = {**best_training_config, "seed": run["seed"]}
    checkpoint = dict(
        model_class="StandaloneCandidateGINE", model_config=run["model_config"].copy(),
        model_state_dict={k: v.detach().cpu() for k, v in run["model"].state_dict().items()},
        training_config=training_config, training_summary=run["training"],
        edge_feature_names=list(graph_edge_builder.feature_names), edge_context="original_16_words",
        embedding_model_name=EMBEDDING_MODEL, feature_metadata=feature_metadata,
        split_dates=split_dates, dataset_path=str(DATASET_PATH.resolve()))
    torch.save(checkpoint, run_weights_path)
    saved_runs.append(dict(seed=run["seed"], checkpoint=str(run_weights_path)))
    if run["seed"] == primary_run["seed"]:
        weights_path = run_weights_path
(RUN_DIR / "saved-models.json").write_text(json.dumps(
    dict(primary_checkpoint=str(weights_path), runs=saved_runs), indent=2) + "\n")
print("Saved primary GINE checkpoint:", weights_path)
print("Saved all fresh-seed checkpoints:", saved_runs)


Saved primary GINE checkpoint: weights/standalone_gine_optuna_text_seed_12_20261026T013804Z.pt
Saved all fresh-seed checkpoints: [{'seed': 10, 'checkpoint': 'weights/standalone_gine_optuna_text_seed_10_20261026T013804Z.pt'}, {'seed': 11, 'checkpoint': 'weights/standalone_gine_optuna_text_seed_11_20261026T013804Z.pt'}, {'seed': 12, 'checkpoint': 'weights/standalone_gine_optuna_text_seed_12_20261026T013804Z.pt'}, {'seed': 13, 'checkpoint': 'weights/standalone_gine_optuna_text_seed_13_20261026T013804Z.pt'}]
